In [1]:
import pandas as pd

df_ground_truth = pd.read_csv('ground_truth-new.csv')

In [2]:
df_ground_truth.head()

,question,document
0,Can I still join the course if I just found ou...,74eb249bbf
1,Is it too late to start this course now?,74eb249bbf
2,"If I join late, can I still get a certificate ...",74eb249bbf
3,What do I need to do to be eligible for the ce...,74eb249bbf
4,Are project submissions still open for new stu...,74eb249bbf


In [3]:
ground_truth = df_ground_truth.to_dict(orient="records")

In [33]:
ground_truth[0]

{'question': 'Can I still join the course if I just found out about it?',
 'document': '74eb249bbf'}

In [5]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm

index = build_index(documents)


In [6]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [7]:
q = ground_truth[0]
q

{'question': 'Can I still join the course if I just found out about it?',
 'document': '74eb249bbf'}

In [8]:
doc_id = q["document"]
results = text_search(query=q["question"])

In [9]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
a9353fadfe == 74eb249bbf: False
9f689c185f == 74eb249bbf: False
5cc511f85b == 74eb249bbf: False
977bf7786c == 74eb249bbf: False


In [10]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [11]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [12]:
q = ground_truth[45]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Where can I find the course lectures or videos for LLM Zoomcamp?


[0, 1, 0, 0, 0]

In [13]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [14]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [15]:
relevance_total_text

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [16]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [17]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [22]:
example_total = compute_relevance_total(ground_truth_sample, text_search)
example_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [19]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/720 [00:00<?, ?it/s]

In [20]:
relevance_total

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0,

In [43]:
def hit_rate(relevance):
    cnt = 0
    for line in relevance:
        if 1 in line:
            
            cnt = cnt+ 1
    return cnt / len(relevance)

In [44]:
hit_rate(example_total)

0.8666666666666667

In [34]:
total_score = 0.0

for line in example_total:
    for rank in range(len(line)):
        if line[rank] == 1:
            score=1/(rank + 1)
            total_score += score
            break

total_score/len(example_total)
    

0.7222222222222221

In [42]:
def mrr(relevance):    
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score=1/(rank + 1)
                total_score += score
                break

    return total_score/len(relevance)

In [36]:
mrr(example_total)

0.7222222222222221

In [37]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [38]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/720 [00:00<?, ?it/s]

606
720


{'hit_rate': 0.8416666666666667, 'mrr': 0.7153703703703698}

In [40]:
def search_boost(query, question_boost=3.0):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [45]:
for boost in [0.5, 1.0,1.5, 2.0, 2.5, 3.0, 3.5, 4.0]:
    print(f"Boost: {boost}")
    print(evaluate(
        ground_truth,
        lambda query: search_boost(query, question_boost=boost)
    ))

Boost: 0.5


  0%|          | 0/720 [00:00<?, ?it/s]

{'hit_rate': 0.9, 'mrr': 0.7983101851851847}
Boost: 1.0


  0%|          | 0/720 [00:00<?, ?it/s]

{'hit_rate': 0.9055555555555556, 'mrr': 0.7903240740740739}
Boost: 1.5


...

{'hit_rate': 0.8416666666666667, 'mrr': 0.7153703703703698}
Boost: 3.5


  0%|          | 0/720 [00:00<?, ?it/s]

{'hit_rate': 0.8361111111111111, 'mrr': 0.7105092592592586}
Boost: 4.0


  0%|          | 0/720 [00:00<?, ?it/s]

{'hit_rate': 0.8263888888888888, 'mrr': 0.7006249999999995}


In [46]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [47]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/720 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/720 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/720 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/720 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/720 [00:00<?, ?it/s]

In [48]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
3,1.0,2.0,0.1,0.975000,0.869931
19,2.0,4.0,0.2,0.975000,0.869931
35,5.0,10.0,0.5,0.975000,0.869931
4,1.0,2.0,0.2,0.972222,0.868426
20,2.0,4.0,0.5,0.969444,0.867269
18,2.0,4.0,0.1,0.969444,0.866759
33,5.0,10.0,0.1,0.966667,0.866597
34,5.0,10.0,0.2,0.968056,0.866481
6,1.0,4.0,0.1,0.965278,0.863819
7,1.0,4.0,0.2,0.970833,0.863449
